# Ray aiming

Step 1: The ray height function

Given:
* An optical sequential system
* A position P in object space
* An angle w.r.t the optical axis theta

The function F(theta) raytraces throught the system and we obtain a ray height at the system's exit joint.

Exercise 1: plot F(theta)


In [ ]:
import torchlensmaker as tlm
import torch
import matplotlib.pyplot as plt

lens = tlm.lenses.singlet(
    tlm.SphereByRadius(8.0, R=10.0),
    tlm.OuterGap(5),
    tlm.Parabola(8.0, A=-0.1, normalize=False),
    #tlm.SphereByRadius(8.0, R=-10.0),
    material="BK7",
    anchors=(0, 0),
)

source = tlm.Sequential(
    tlm.SubChain(
        tlm.Gap(-10),
        tlm.PointSource(50),
    )
)

tlm.set_sampling2d(source, pupil=100)
inputs = source.sequential(tlm.default_input(dim=2))
outputs = lens.sequential(inputs)

tlm.show2d(tlm.Sequential(
    source,
    lens,
), end=5)

# theta is the angle of input ray with the optical axis
thetas = torch.rad2deg(torch.atan2(inputs.rays.V[:, 1], inputs.rays.V[:, 0]))

# ray height at system exit
ray_height = outputs.rays.V[:, 1]

# create input ray from theta
#   - what pupil/field/wavel coordinates?
# evaluate model forward
# obtain ray height

f, ax = plt.subplots(1, 1)
ax.plot(thetas.detach().numpy(), ray_height.detach().numpy(), linestyle="-", marker="none")
ax.set_xlabel("input angle (deg)")
ax.set_ylabel("output ray height")

Exercise 2: The raytracing approach

Make the aperture stop into an light source
raytrace it throught the reverse entrance system
i.e. the surfaces from the front up until the aperture stop, in retrograde direction

Procedure: fit an image plane / image surface

Given an optical system (source + lens) where is the image? formed?

In [ ]:
import torch.optim as optim
import torchlensmaker as tlm
import torch

d1, d2 = 30, 25

r1 = tlm.SphereByCurvature(d1, 1 / 26.722)
r2 = tlm.SphereByCurvature(d1, 1 / -151.086)
r3 = tlm.SphereByCurvature(d2, 1 / -29.485)
r4 = tlm.SphereByCurvature(d2, 1 / 24.273)
r5 = tlm.SphereByCurvature(d1, 1 / 150.388)
r6 = tlm.SphereByCurvature(d1, 1 / -26.407)

material1 = tlm.NonDispersiveMaterial(1.5108)
material2 = tlm.NonDispersiveMaterial(1.6042)

L1 = tlm.lenses.singlet(r1, tlm.InnerGap(5.9), r2, material=material1)
L2 = tlm.lenses.singlet(r3, tlm.InnerGap(0.2), r4, material=material2)
L3 = tlm.lenses.singlet(r5, tlm.InnerGap(5.9), r6, material=material1)

focal = tlm.parameter(84.4495)

optics = tlm.Sequential(
    #tlm.ObjectAtInfinity(15, 25),
    L1,
    tlm.Gap(10.9),
    L2,
    tlm.Gap(3.1),
    # tlm.Aperture(18),
)

# shouldn't object diameter be the first argument?


# Now we want to find the position of the image plane,
# which will be our entrance pupil

static_lens = optics.clone()
for p in static_lens.parameters():
    p.requires_grad = False

epmodel = tlm.Sequential(
    tlm.Reversed(tlm.Object(60, object_diameter=18)),
    tlm.Reversed(static_lens),
    tlm.Gap(0, trainable=True),
    tlm.ImagePlane(60),
)

for name, p in epmodel.named_parameters():
    print(name, p.item(), p.requires_grad)

epmodel.set_sampling2d(pupil=20, field=20, wavel=1)

tlm.optimize(epmodel,
             dim=2,
             optimizer = optim.RMSprop(epmodel.parameters(), lr=1),
             num_iter=50,
).plot()


In [ ]:
epmodel.set_sampling2d(pupil=50, field=3, wavel=1)
tlm.show2d(epmodel, end=40)

In [ ]:
# EP diameter
head, image_plane = epmodel[:-1], epmodel[-1]
rays = head.sequential(tlm.default_input(dim=2))
rays_image, _ = image_plane(rays.rays, rays.fk, rays.direction)

print(rays_image.shape)
ep_diameter = rays_image.max() * 2
ep_x = tlm.hom_target(rays.fk.direct)[0]

print("EP diameter", ep_diameter)
print("EP X coordinate", ep_x)  # wrong reference frame